# Задание 3. Модульные ёмкости и маршрутизация
## 1. Постановка и теоретические основания
Задано: направленный граф $G=(N,E)$, матрица трафика $h_{nt}$, ёмкость одного модуля $C>0$, длины каналов $d_e>0$.
Найти число модулей $a_e\in\mathbb Z_{\ge0}$ и агрегированные по назначению потоки $x_{te}\ge0$.

Два самостоятельных критерия из ТЗ:
$$F_1=\min\sum_e a_e,\qquad F_2=\min\sum_e d_ea_e.$$
Ограничения:
$$\sum_{e\in\delta^+(n)}x_{te}-\sum_{e\in\delta^-(n)}x_{te}=\begin{cases}h_{nt},&n\ne t,\\-\sum_{s\ne t}h_{st},&n=t,\end{cases}$$
$$\sum_t x_{te}\le Ca_e.$$
**Основание:** main.pdf, задание 3, стр. 7–10; формулировка «адрес назначения — канал», §4.4, стр. 46–52.

### Почему MILP
Целевая функция и ограничения линейны. Модули целочисленны, потоки непрерывны. Это смешанно-целочисленная линейная оптимизация (MILP). Нельзя решать непрерывную модель и затем округлять модули: округление не гарантирует оптимальности и допустимости.

### Физический смысл
Баланс означает: узел пересылает входящий трафик плюс собственный; назначение поглощает суммарный трафик. На канале суммируются потоки ко всем назначениям. Резерв ёмкости допустим вследствие неделимости модулей.

### Допущения
Трафик делим; $h_{tt}=0$; модули не имеют верхнего предела. Ненаправленный канал в данных раскрывается в пару противоположных дуг с одной длиной. Параллельные каналы не поддерживаются. Объёмы и $C$ — в условных единицах, длина — км, второй критерий — модуль·км.

Модель не включает ECMP, задержки и отказоустойчивость. Текущая матрица трафика не определяет универсальные таблицы маршрутизации для всех будущих требований.

### Что обязательно по ТЗ и что добавлено
| Блок | Статус |
|------|--------|
| Два критерия $F_1$, $F_2$ по отдельности | ТЗ |
| Таблица загрузки, доли $f_{te}$ | ТЗ / §4.4.1 |
| Визуализация, экспорт JSON/CSV | ТЗ (в ноутбуке) |
| Лексикография: min $F_2$ при фиксации оптимального $F_1$ | Добавлено (последовательная оптимизация) |
| Устранение циркуляции при фиксированных модулях | Добавлено (§4.4.2) |


## 2. Окружение и способ запуска
При необходимости выполните отдельно:
```python
%pip install pyomo highspy scipy pandas numpy networkx matplotlib
```
Решатель: `SolverFactory('appsi_highs')` (нужен пакет `highspy`).


In [ ]:
import copy, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pyomo.environ as pyo
from IPython.display import display as _display, HTML

def display(obj):
    _display(HTML(obj.to_html(index=False))) if isinstance(obj, pd.DataFrame) else _display(obj)

ATOL, RTOL = 1e-7, 1e-7
EXPORT_DIR = Path('exports_modular_capacity')

def tolerance(*values):
    return ATOL + RTOL * max([1.0] + [abs(float(v)) for v in values])


## 3. Данные и проверка достижимости
Ненаправленные каналы задаются ключом `undirected`: каждая пара раскрывается в две противоположные дуги с одной длиной. Проверка путей даёт раннее сообщение, окончательный вердикт о допустимости — у решателя.


In [ ]:
def expand_links(d):
    # directed + undirected -> пара противоположных дуг с одной длиной
    links = dict(d.get('links', {}))
    for (u, v), length in d.get('undirected', {}).items():
        if u == v:
            raise ValueError('Петля недопустима')
        for a, b in ((u, v), (v, u)):
            if (a, b) in links and abs(links[(a, b)] - length) > ATOL:
                raise ValueError(f'Конфликт длины для {(a, b)}')
            links[(a, b)] = length
    return links

def normalize_data(d):
    out = copy.deepcopy(d)
    out['links'] = expand_links(out)
    out.pop('undirected', None)
    return out

def validate_data(d):
    d = normalize_data(d)
    N = d['nodes']
    if not N or len(N) != len(set(N)):
        raise ValueError('Нужны непустые уникальные узлы')
    if not math.isfinite(d['C']) or d['C'] <= 0:
        raise ValueError('C должно быть конечным и положительным')
    if not d['links']:
        raise ValueError('Не заданы каналы')
    for (u, v), length in d['links'].items():
        if u not in N or v not in N or u == v:
            raise ValueError('Некорректный канал')
        if not math.isfinite(length) or length <= 0:
            raise ValueError('Некорректная длина')
    G = nx.DiGraph()
    G.add_nodes_from(N)
    G.add_edges_from(d['links'])
    unreachable = []
    for (s, t), h in d['demands'].items():
        if s not in N or t not in N or s == t:
            raise ValueError('Некорректное требование')
        if not math.isfinite(h) or h < 0:
            raise ValueError('Некорректный трафик')
        if h > 0 and not nx.has_path(G, s, t):
            unreachable.append((s, t))
    return unreachable

# Демо-пример (в PDF числовой инстанс не задан).
data = {
    'nodes': ['A', 'B', 'C', 'D', 'E'],
    'links': {
        ('A', 'D'): 12, ('A', 'B'): 2, ('B', 'C'): 2, ('C', 'D'): 2,
        ('B', 'D'): 9, ('E', 'D'): 10, ('E', 'C'): 2, ('A', 'E'): 4,
    },
    'demands': {('A', 'D'): 8, ('E', 'D'): 6, ('B', 'D'): 3},
    'C': 5.0,
}
print('Недостижимые требования:', validate_data(data))
display(pd.DataFrame(
    [(s, t, h) for (s, t), h in data['demands'].items()],
    columns=['source', 'destination', 'traffic'],
))


## 4. Модель Pyomo
Целочисленность задаётся до решения. Для релаксации меняется только область значений модулей. Индексы потока: назначение $t$, затем начало и конец канала.


In [ ]:
def build_model(d, mode='modules', relax=False):
    d = normalize_data(d)
    validate_data(d)
    if mode not in ('modules', 'distance'):
        raise ValueError('Неизвестный критерий')
    m = pyo.ConcreteModel('Destination-link modular capacity')
    m.N = pyo.Set(initialize=d['nodes'], ordered=True)
    m.E = pyo.Set(initialize=list(d['links']), dimen=2, ordered=True)
    m.C = pyo.Param(initialize=d['C'])
    m.length = pyo.Param(m.E, initialize=d['links'])
    m.h = pyo.Param(m.N, m.N, initialize=d['demands'], default=0)
    m.a = pyo.Var(m.E, domain=pyo.NonNegativeReals if relax else pyo.NonNegativeIntegers)
    m.x = pyo.Var(m.N, m.E, domain=pyo.NonNegativeReals)
    outgoing = {n: [e for e in m.E if e[0] == n] for n in m.N}
    incoming = {n: [e for e in m.E if e[1] == n] for n in m.N}

    def balance(m, t, n):
        rhs = m.h[n, t] if n != t else -sum(m.h[s, t] for s in m.N if s != t)
        if not outgoing[n] and not incoming[n]:
            return pyo.Constraint.Feasible if pyo.value(rhs) == 0 else pyo.Constraint.Infeasible
        return (
            sum(m.x[t, u, v] for u, v in outgoing[n])
            - sum(m.x[t, u, v] for u, v in incoming[n])
            == rhs
        )

    m.balance = pyo.Constraint(m.N, m.N, rule=balance)
    m.capacity = pyo.Constraint(
        m.E,
        rule=lambda m, u, v: sum(m.x[t, u, v] for t in m.N) <= m.C * m.a[u, v],
    )
    m.module_count = pyo.Expression(expr=sum(m.a[e] for e in m.E))
    m.distance_cost = pyo.Expression(expr=sum(m.length[e] * m.a[e] for e in m.E))
    m.total_flow = pyo.Expression(expr=sum(m.x[t, u, v] for t in m.N for u, v in m.E))
    m.objective = pyo.Objective(
        expr=m.module_count if mode == 'modules' else m.distance_cost,
        sense=pyo.minimize,
    )
    m._mode = mode
    return m


## 5. Решатель и сертификат завершения
Допустимость не равна оптимальности. В минимизации допустимое решение даёт верхнюю оценку, поиск/релаксации — нижнюю. Статус и граница выводятся отдельно.


In [ ]:
def solve_model(m):
    solver = pyo.SolverFactory('appsi_highs')
    if not solver.available(exception_flag=False):
        raise RuntimeError('Нужен highspy: %pip install highspy; перезапустите ядро')
    solver.options['time_limit'] = 60
    solver.options['mip_rel_gap'] = 0.0
    solver.options['mip_abs_gap'] = 0.0
    result = solver.solve(m, load_solutions=False)
    term = result.solver.termination_condition
    status = (
        'optimal' if term == pyo.TerminationCondition.optimal else
        'infeasible' if term == pyo.TerminationCondition.infeasible else
        str(term)
    )
    if status == 'optimal':
        m.solutions.load_from(result)

    def finite_or_none(value):
        try:
            z = float(value)
            return z if math.isfinite(z) else None
        except (TypeError, ValueError):
            return None

    bound = finite_or_none(result.problem.lower_bound)
    objective = pyo.value(m.objective) if status == 'optimal' else None
    gap = (
        abs(objective - bound) / max(abs(objective), 1e-10)
        if objective is not None and bound is not None else None
    )
    return {
        'status': status,
        'backend': 'Pyomo/appsi_highs',
        'objective': objective,
        'best_bound': bound,
        'relative_gap': gap,
        'termination': str(term),
    }


## 6. Проверка решения, таблицы и циклы
Используется абсолютный плюс относительный допуск. Целочисленность проверяется отдельно. Округление в таблице не используется для проверки допустимости.


In [ ]:
def check_solution(m):
    for v in m.component_data_objects(pyo.Var):
        z = pyo.value(v)
        if not math.isfinite(z):
            raise ValueError(f'Неконечное значение {v.name}')
        if v.lb is not None and z < pyo.value(v.lb) - tolerance(z, v.lb):
            raise ValueError(v.name)
        if v.ub is not None and z > pyo.value(v.ub) + tolerance(z, v.ub):
            raise ValueError(v.name)
        if v.is_integer() and abs(z - round(z)) > 1e-5:
            raise ValueError(f'Не целое: {v.name}')
    for con in m.component_data_objects(pyo.Constraint, active=True):
        z = pyo.value(con.body)
        if not math.isfinite(z):
            raise ValueError(con.name)
        for bound, sign in ((con.lower, -1), (con.upper, 1)):
            if bound is not None and sign * (z - pyo.value(bound)) > tolerance(z, pyo.value(bound)):
                raise ValueError(f'Нарушение {con.name}')
    return True

def tables(m):
    rows = []
    for u, v in m.E:
        a = pyo.value(m.a[u, v])
        flow = sum(pyo.value(m.x[t, u, v]) for t in m.N)
        capacity = pyo.value(m.C) * a
        rows.append([
            u, v, pyo.value(m.length[u, v]), a, capacity, flow,
            flow / capacity if capacity > ATOL else np.nan,
            capacity - flow,
        ])
    links = pd.DataFrame(
        rows,
        columns=['source', 'target', 'length', 'modules', 'capacity', 'flow', 'utilization', 'reserve'],
    )
    flows = pd.DataFrame(
        [
            (t, u, v, pyo.value(m.x[t, u, v]))
            for t in m.N for u, v in m.E
            if pyo.value(m.x[t, u, v]) > ATOL
        ],
        columns=['destination', 'source', 'target', 'flow'],
    )
    return links, flows

def summary(m):
    return {
        'modules': pyo.value(m.module_count),
        'weighted_cost': pyo.value(m.distance_cost),
        'total_link_flow': pyo.value(m.total_flow),
    }

def cycle_report(m):
    report = {}
    for t in m.N:
        G = nx.DiGraph()
        G.add_nodes_from(m.N)
        G.add_edges_from([(u, v) for u, v in m.E if pyo.value(m.x[t, u, v]) > ATOL])
        report[t] = not nx.is_directed_acyclic_graph(G)
    return report

def run(d, mode='modules', relax=False):
    m = build_model(d, mode, relax)
    info = solve_model(m)
    if info['status'] == 'optimal':
        check_solution(m)
    return m, info


## 7. Два критерия ТЗ и линейная релаксация
Ниже — **обязательные** расчёты задания: отдельно $F_1=\min\sum a_e$ и отдельно $F_2=\min\sum d_e a_e$.

Линейная релаксация допускает дробные модули, поэтому $z_{LP}\le z_{MILP}$. Это нижняя оценка, а не допустимая установка оборудования.


In [ ]:
tz_results = []
models = {}
certificates = []
comparison = []

for mode in ('modules', 'distance'):
    m, info = run(data, mode)
    assert info['status'] == 'optimal', info
    lp, lpinfo = run(data, mode, True)
    assert lpinfo['status'] == 'optimal', lpinfo
    assert pyo.value(lp.objective) <= pyo.value(m.objective) + tolerance(pyo.value(m.objective))
    models[mode] = m
    certificates.append({'scope': 'ТЗ: отдельный критерий', 'mode': mode, **info})
    comparison.append({'mode': mode, **summary(m), 'LP_bound': pyo.value(lp.objective)})
    print('=== ТЗ /', mode, '===')
    links, flows = tables(m)
    display(links.round(5))
    display(flows.round(5))

display(pd.DataFrame(certificates))
display(pd.DataFrame(comparison))

tz_results.append({'criterion': 'F1 min modules', 'value': pyo.value(models['modules'].objective)})
tz_results.append({'criterion': 'F2 min distance cost', 'value': pyo.value(models['distance'].objective)})
display(pd.DataFrame(tz_results))

assert abs(pyo.value(models['modules'].objective) - 5) < 1e-6
assert abs(pyo.value(models['distance'].objective) - 22) < 1e-6
assert abs(comparison[0]['LP_bound'] - 3.4) < 1e-6
assert abs(comparison[1]['LP_bound'] - 16.8) < 1e-6


## 8. Добавлено: выбор среди оптимумов и устранение циркуляции
В критерии $F_1$ длины не участвуют, поэтому среди решений с минимальным числом модулей стоимость в модуль·км может отличаться.

**Дополнительный этап (не подмена ТЗ):** фиксируем оптимальное $F_1$ и минимизируем $F_2$ (последовательная оптимизация с нулевой уступкой). Исходный оптимум по модулям сохраняется.

Далее фиксируем установленные модули и минимизируем суммарный поток — устраняем положительную циркуляцию (§4.4.2).


In [ ]:
def change_objective(m, expr):
    m.del_component(m.objective)
    m.objective = pyo.Objective(expr=expr, sense=pyo.minimize)

def remove_circulation(m):
    for e in m.E:
        m.a[e].fix(round(pyo.value(m.a[e])))
    change_objective(m, m.total_flow)
    info = solve_model(m)
    if info['status'] != 'optimal':
        raise RuntimeError(info)
    check_solution(m)
    assert not any(cycle_report(m).values()), cycle_report(m)
    return info

lex = models['modules'].clone()
optimal_modules = round(pyo.value(lex.module_count))
lex.keep_modules = pyo.Constraint(expr=lex.module_count == optimal_modules)
change_objective(lex, lex.distance_cost)
lex_info = solve_model(lex)
assert lex_info['status'] == 'optimal'
check_solution(lex)
assert abs(pyo.value(lex.distance_cost) - 45) < 1e-6
clean_info = remove_circulation(lex)

distance_clean = models['distance'].clone()
remove_circulation(distance_clean)

extra = pd.DataFrame([
    {
        'scope': 'Добавлено: лексикография F1→F2',
        'modules': pyo.value(lex.module_count),
        'weighted_cost': pyo.value(lex.distance_cost),
        **{k: lex_info[k] for k in ('status', 'backend', 'objective')},
    },
    {
        'scope': 'Добавлено: очистка циркуляции',
        'modules': pyo.value(lex.module_count),
        'weighted_cost': pyo.value(lex.distance_cost),
        'total_link_flow': pyo.value(lex.total_flow),
        **{k: clean_info[k] for k in ('status', 'backend')},
    },
])
print('Вторичный оптимум при пяти модулях:', summary(lex))
display(tables(lex)[0].round(5))
print('Циклы после очистки:', cycle_report(lex))
display(extra)


## 9. Доли пересылки
По §4.4.1, формула (4.19):
$$f_{te}=\frac{x_{te}}{h_{a(e)t}+\sum_{e'\in\delta^-(a(e))}x_{te'}}.$$
Для узла с положительным доступным трафиком сумма долей по исходящим каналам равна единице. В назначении трафик поглощается. При нулевом знаменателе — NaN: текущие требования не определяют правило.


In [ ]:
def forwarding_table(m):
    rows = []
    for t in m.N:
        for n in m.N:
            if n == t:
                continue
            available = pyo.value(m.h[n, t]) + sum(
                pyo.value(m.x[t, u, v]) for u, v in m.E if v == n
            )
            outgoing = [(u, v) for u, v in m.E if u == n]
            fractions = []
            for u, v in outgoing:
                flow = pyo.value(m.x[t, u, v])
                fraction = flow / available if available > ATOL else np.nan
                rows.append([t, n, v, available, flow, fraction])
                fractions.append(fraction)
            if available > ATOL:
                assert abs(sum(fractions) - 1) <= tolerance(sum(fractions), 1)
    return pd.DataFrame(
        rows,
        columns=['destination', 'node', 'next_node', 'available', 'flow', 'fraction'],
    )

ft_lex = forwarding_table(lex)
display(ft_lex.round(5))


## 10. Визуализация
Сравниваются: оптимум ТЗ по модулям, лексикографический выбор и оптимум по стоимости после очистки потоков.

- Толщина ребра пропорциональна числу модулей $a_e$.
- Подпись: поток / ёмкость (utilization).
- Отдельный ряд: потоки, раскрашенные по назначению $t$.


In [ ]:
pos = {'A': (0, 1), 'B': (1, 1), 'C': (2, 0.5), 'D': (3, 1), 'E': (1, -0.5)}
G = nx.DiGraph()
G.add_nodes_from(data['nodes'])
G.add_edges_from(normalize_data(data)['links'])

def draw_modules(ax, m, title):
    table, _ = tables(m)
    active = table[table.modules > ATOL]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightblue', node_size=650)
    nx.draw_networkx_labels(G, pos, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='lightgray', style='dashed', node_size=650, alpha=0.5)
    if active.empty:
        ax.set_title(title)
        ax.axis('off')
        return
    widths = [1.0 + 2.5 * float(r.modules) for r in active.itertuples()]
    edgelist = list(zip(active.source, active.target))
    nx.draw_networkx_edges(
        G, pos, ax=ax, edgelist=edgelist, width=widths,
        edge_color='steelblue', node_size=650, arrows=True, arrowstyle='-|>',
    )
    labels = {
        (r.source, r.target): f'{r.flow:g}/{r.capacity:g}\n({r.utilization:.0%})'
        for r in active.itertuples()
    }
    nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=labels, font_size=7)
    ax.set_title(title)
    ax.axis('off')

def draw_flows_by_destination(ax, m, title):
    from matplotlib.lines import Line2D
    _, flows = tables(m)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightblue', node_size=650)
    nx.draw_networkx_labels(G, pos, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='lightgray', style='dashed', node_size=650, alpha=0.4)
    destinations = sorted(flows.destination.unique()) if not flows.empty else []
    cmap = cm.get_cmap('tab10', max(len(destinations), 1))
    handles = []
    for i, t in enumerate(destinations):
        sub = flows[flows.destination == t]
        edgelist = list(zip(sub.source, sub.target))
        widths = [1.0 + 1.5 * float(f) for f in sub.flow]
        color = cmap(i)
        nx.draw_networkx_edges(
            G, pos, ax=ax, edgelist=edgelist, width=widths,
            edge_color=[color], node_size=650, arrows=True, arrowstyle='-|>',
            connectionstyle=f'arc3,rad={0.08 * (i - (len(destinations) - 1) / 2)}',
        )
        handles.append(Line2D([0], [0], color=color, lw=2, label=f't={t}'))
    if handles:
        ax.legend(handles=handles, loc='best', fontsize=8)
    ax.set_title(title)
    ax.axis('off')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
panels = [
    (models['modules'], 'ТЗ: min modules'),
    (lex, 'Добавлено: min cost | 5 modules'),
    (distance_clean, 'ТЗ: min distance cost'),
]
for ax, (m, title) in zip(axes[0], panels):
    draw_modules(ax, m, title)
for ax, (m, title) in zip(axes[1], panels):
    draw_flows_by_destination(ax, m, 'Потоки по t: ' + title.split(':')[-1].strip())
plt.tight_layout()
plt.show()


## 11. Экспорт результатов (JSON / CSV)
Сохраняются модули, потоки $x_{te}$, загрузка каналов и доли пересылки — для выбранного решения (лексикографический оптимум после очистки).


In [ ]:
def export_solution(m, stem, directory=EXPORT_DIR):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    links, flows = tables(m)
    fwd = forwarding_table(m)
    payload = {
        'summary': summary(m),
        'C': pyo.value(m.C),
        'links': links.replace({np.nan: None}).to_dict(orient='records'),
        'flows': flows.to_dict(orient='records'),
        'forwarding': fwd.replace({np.nan: None}).to_dict(orient='records'),
    }
    json_path = directory / f'{stem}.json'
    json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    links.to_csv(directory / f'{stem}_links.csv', index=False)
    flows.to_csv(directory / f'{stem}_flows.csv', index=False)
    fwd.to_csv(directory / f'{stem}_forwarding.csv', index=False)
    return {
        'json': str(json_path),
        'links_csv': str(directory / f'{stem}_links.csv'),
        'flows_csv': str(directory / f'{stem}_flows.csv'),
        'forwarding_csv': str(directory / f'{stem}_forwarding.csv'),
    }

export_paths = export_solution(lex, 'lex_clean')
display(pd.DataFrame([export_paths]))
assert Path(export_paths['json']).exists()
assert Path(export_paths['links_csv']).exists()


## 12. Автоматические тесты
Проверяются ожидаемые значения цели, ограничения, некорректный ввод, недостижимость, агрегация, undirected-пары, устранение циркуляции и экспорт.


In [ ]:
tests = []

def passed(name):
    tests.append({'test': name, 'result': 'PASS'})

def test_value(name, d, expected, mode='modules'):
    m, info = run(d, mode)
    assert info['status'] == 'optimal', info
    assert abs(pyo.value(m.objective) - expected) < 1e-6
    assert check_solution(m)
    passed(name)

small = {'nodes': ['S', 'T'], 'links': {('S', 'T'): 2}, 'demands': {('S', 'T'): 12}, 'C': 5}
test_value('Три модуля для 12 единиц', small, 3)
test_value('Стоимость 3 × 2', small, 6, 'distance')

zero = copy.deepcopy(small)
zero['demands'] = {}
test_value('Нулевой трафик', zero, 0)

agg = {
    'nodes': ['A', 'B', 'M', 'T'],
    'links': {('A', 'M'): 1, ('B', 'M'): 1, ('M', 'T'): 1},
    'demands': {('A', 'T'): 4, ('B', 'T'): 4},
    'C': 5,
}
test_value('Агрегация: 1+1+2', agg, 4)

# Ненаправленный канал раскрывается в пару дуг.
und = {
    'nodes': ['S', 'T'],
    'links': {},
    'undirected': {('S', 'T'): 3},
    'demands': {('S', 'T'): 7},
    'C': 5,
}
assert set(normalize_data(und)['links']) == {('S', 'T'), ('T', 'S')}
test_value('Undirected: модули только в нужную сторону', und, 2)
mu, _ = run(und)
assert round(pyo.value(mu.a['S', 'T'])) == 2
assert round(pyo.value(mu.a['T', 'S'])) == 0
passed('Undirected: обратная дуга без модулей')

bad = {'nodes': ['S', 'T'], 'links': {('T', 'S'): 1}, 'demands': {('S', 'T'): 1}, 'C': 5}
assert validate_data(bad) == [('S', 'T')]
_, info = run(bad)
assert info['status'] == 'infeasible', info
passed('Нет пути: статус решателя')

for field, value in [('C', 0), ('C', float('nan'))]:
    d = copy.deepcopy(small)
    d[field] = value
    try:
        validate_data(d)
    except ValueError:
        passed(f'Отклонение {field}={value}')
    else:
        raise AssertionError('Некорректный ввод принят')

cyclic = {
    'nodes': ['S', 'A', 'T'],
    'links': {('S', 'A'): 1, ('A', 'S'): 1, ('A', 'T'): 1},
    'demands': {('S', 'T'): 1},
    'C': 5,
}
cm = build_model(cyclic)
for e in cm.E:
    cm.a[e].fix(1)
for t in cm.N:
    for u, v in cm.E:
        cm.x[t, u, v].set_value(0)
cm.x['T', 'S', 'A'].set_value(2)
cm.x['T', 'A', 'S'].set_value(1)
cm.x['T', 'A', 'T'].set_value(1)
assert check_solution(cm) and cycle_report(cm)['T']
passed('Допустимость не исключает циркуляцию')
remove_circulation(cm)
assert not any(cycle_report(cm).values())
passed('Удаление циркуляции')

assert check_solution(lex) and abs(pyo.value(lex.distance_cost) - 45) < 1e-6
passed('Вторичный оптимум 45')
assert not forwarding_table(lex).empty
passed('Доли пересылки')
for mode, m in models.items():
    assert check_solution(m)
    passed('Баланс и ёмкости: ' + mode)

paths = export_solution(models['distance'], 'tz_distance')
assert Path(paths['json']).exists() and Path(paths['flows_csv']).exists()
passed('Экспорт JSON/CSV')

# Сумма долей = 1 для узлов с положительным available.
ft = forwarding_table(lex)
for (t, n), g in ft.groupby(['destination', 'node']):
    avail = float(g.available.iloc[0])
    if avail > ATOL:
        assert abs(g.fraction.sum() - 1) <= tolerance(1)
passed('Инвариант суммы f_te')

display(pd.DataFrame(tests))


## 13. Эксперименты
Меняется только один фактор. Побочные метрики относятся к конкретному найденному оптимуму.


In [ ]:
experiments = []
for series, values in [('capacity', [3, 5, 10]), ('traffic', [1, 1.5, 2])]:
    for factor in values:
        for mode in ['modules', 'distance']:
            d = copy.deepcopy(data)
            if series == 'capacity':
                d['C'] = factor
            else:
                d['demands'] = {pair: h * factor for pair, h in data['demands'].items()}
            m, info = run(d, mode)
            assert info['status'] == 'optimal', info
            experiments.append({
                'series': series, 'factor': factor, 'mode': mode,
                **summary(m), 'backend': info['backend'],
            })
exp = pd.DataFrame(experiments)
display(exp)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, series, label in zip(axes, ['capacity', 'traffic'], ['Module capacity', 'Traffic multiplier']):
    for mode in ['modules', 'distance']:
        sub = exp[(exp.series == series) & (exp['mode'] == mode)]
        ax.plot(sub.factor, sub.modules, 'o-', label=mode)
    ax.set_xlabel(label)
    ax.set_ylabel('Modules')
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()


## 14. Выводы и ограничения верификации
### Обязательные результаты ТЗ
- Минимум модулей ($F_1$): **5**.
- Минимум стоимости ($F_2$): **22** модуль·км.

### Добавленные этапы
- При сохранении минимума в 5 модулей лучший вторичный критерий: **45** модуль·км (значение 53 у «первого» оптимума по модулям допустимо, но не вторичный оптимум).
- Релаксации: 3.4 модуля и 16.8 модуль·км — нижние оценки, не план установки.
- Циркуляция устраняется отдельным этапом при фиксированных модулях.
- Доли пересылки — для текущих потоков; правила для неиспользуемых назначений остаются неопределёнными.
- Экспорт: каталог `exports_modular_capacity/`.

Полный интерактивный редактор топологии (React/Leaflet + API + БД) в это решение не входит: матмодель и отчётность реализованы в ноутбуке.

### Источники
1. main.pdf: задание 3, стр. 7–10; §4.4–4.4.2, стр. 46–52.
2. Пособие по СППР: MILO, границы, последовательная оптимизация.


In [ ]:
print(f'Пройдено тестов: {len(tests)}; экспериментальных сценариев: {len(exp)}')
print('Backend:', 'Pyomo/appsi_highs')
print('Экспорт:', export_paths['json'])
print('Все ячейки выполнены; проверки завершены без ошибок.')
